In [5]:
# =============================================================================
# HOW DO WE GRADE A FAKE PHOTO? — IS, FID, CLIP Score
# =============================================================================
#
# GAN / VAE / diffusion can all spit out pictures. The human test is easy:
#   "Does this look like a real cat?"
# Machines need a NUMBER, because we cannot eyeball 50,000 fakes.
#
# Pixel MSE is a bad grade for art. Two different photos of "a red cat"
# can have huge pixel error and still both be good. So we grade in the
# BRAIN of a frozen classifier (Inception or CLIP), not in raw pixels.
#
#
# ---------------------------------------------------------------------------
# 1) INCEPTION SCORE (IS) — "is it a clear SOMETHING, and many kinds?"
# ---------------------------------------------------------------------------
#
# Pretend Inception-v3 is a museum guard who names ImageNet classes
# (1000 labels: tabby, sports car, …). You show it a pile of fakes.
#
# For ONE image it outputs p(y | image) — 1000 probabilities.
#
# Two wishes, in English:
#
#   (A) SHARP / CONFIDENT
#       One fake should look like ONE class, not "maybe cat, maybe toaster."
#       Guard says: p(y | image) is PEEKED (one big number, rest tiny).
#
#   (B) DIVERSE
#       The whole pile should cover MANY classes, not 10,000 identical cats.
#       Guard's average guess p(y) should be spread out, not one spike.
#
# IS combines them with KL divergence, then exp():
#
#   IS = exp(  average over images of  KL( p(y|image)  ||  p(y) )  )
#
#   High IS  → each image is a confident class AND the set is varied.
#   Low IS   → mush, or 10,000 copies of the same face.
#
# Rough scale (ImageNet-trained Inception, big sets):
#   random noise ~ 1     CIFAR-ish GANs ~ 2–8     strong ImageNet models >> 10
#
# What IS cannot see:
#   • It never looks at REAL photos. A generator that only makes "perfect
#     ImageNet posters" can score high even if it missed your dataset.
#   • It only knows 1000 ImageNet labels. "Van Gogh fruit bowl" is not a class.
#   • Mode collapse can still sneak through if those few modes look confident.
#
#
# ---------------------------------------------------------------------------
# 2) FRÉCHET INCEPTION DISTANCE (FID) — "do fakes live in the same
#    neighborhood as REAL photos?"
# ---------------------------------------------------------------------------
#
# This is the usual paper number. Lower is better (a DISTANCE).
#
# Picture two clouds of points:
#   Real photos  → run through Inception, grab a mid-layer vector
#                  (not the 1000 labels — a 2048-D "smell" of the image)
#   Fake photos  → same
#
# Fit a blob (Gaussian) to each cloud: mean μ and covariance Σ.
# FID = how far those two blobs are (Fréchet / Wasserstein-2 between Gaussians):
#
#   FID = ||μ_real − μ_fake||²  +  trace( Σ_real + Σ_fake − 2√(Σ_real Σ_fake) )
#
# English:
#   first term  = "are the averages in the same place?"
#   second term = "are the spreads the same shape?"
#
#   FID ≈ 0     fakes indistinguishable from reals (in this feature space)
#   FID small   good (CIFAR papers brag about low tens or less)
#   FID huge    fakes in a different neighborhood (blur, wrong colors, collapse)
#
# Why better than IS for "does it match my data?":
#   FID USES the real set. IS does not.
#
# Gotchas:
#   • Need LOTS of images (papers use 10k–50k). A handful of fakes = noisy FID.
#   • Same Inception preprocess (299×299, ImageNet mean/std) or numbers lie.
#   • Not a human. Two clouds can match while pictures still look weird.
#
#
# ---------------------------------------------------------------------------
# 3) CLIP SCORE — "does the picture MATCH THE PROMPT?"
# ---------------------------------------------------------------------------
#
# IS/FID never read your text. A beautiful mountain when you asked for
# "a cat" still can look "real." CLIP Score grades TEXT–IMAGE agreement.
#
# CLIP (from the Stable Diffusion notebook) has two frozen encoders:
#   image → vector     text → vector     (same vector space)
#
# CLIP Score ≈ cosine similarity (or a scaled version) between:
#   CLIP(image)  and  CLIP("a watercolor cat on a windowsill")
#
#   High  → picture and words point the same way
#   Low   → pretty image, wrong homework
#
# Use it for text-to-image (Stable Diffusion). IS/FID still useful for
# "looks like ImageNet / CIFAR," not for "obeyed the prompt."
#
#
# ---------------------------------------------------------------------------
# One table
# ---------------------------------------------------------------------------
#   Metric     Asks                         Needs reals?   Higher/lower better?
#   -------    ---------------------------  -------------   ---------------------
#   IS         clear + diverse classes      no             HIGHER
#   FID        fake cloud ≈ real cloud      YES            LOWER
#   CLIP       image matches the prompt     no (needs text) HIGHER
#
# Next cells: load Inception, compute IS, then FID, then CLIP on toy images.
#


In [ ]:
# Setup

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torchvision.models import inception_v3, Inception_V3_Weights
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import numpy as np
from scipy import linalg
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seed
torch.manual_seed(42)

In [ ]:
# =============================================================================
# INCEPTION SCORE — code
# =============================================================================
# transform_input=False: WE resize to 299×299 and use ImageNet mean/std.
# (If transform_input=True, Inception would try to undo a different scale.)
#

def load_inception():
    model = inception_v3(weights=Inception_V3_Weights.DEFAULT, transform_input=False)
    model.eval()
    model.to(device)
    return model


inception = load_inception()

inception_transform = transforms.Compose(
    [
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# Now implement the Inception Score. We'll compute probabilities in batches, then calculate the score.
def inception_score(images, batch_size=32, splits=10):
    """
    images: list of PIL Images or a tensor of shape [N, 3, H, W] in [0,1] range.
    Returns mean and std of IS over splits.
    """
    if isinstance(images, torch.Tensor):
        # Assume images are already preprocessed and on device
        pass
    else:
        # Convert list of PIL to tensor
        tensors = []
        for img in images:
            img_tensor = inception_transform(img).unsqueeze(0)
            tensors.append(img_tensor)
        images = torch.cat(tensors, dim=0)
    images = images.to(device)

    N = images.size(0)
    preds = []
    with torch.no_grad():
        for i in range(0, N, batch_size):
            batch = images[i : i + batch_size]
            out = inception(batch)
            logits = out.logits if hasattr(out, "logits") else out
            probs = F.softmax(logits, dim=1)
            preds.append(probs.cpu())
    preds = torch.cat(preds, dim=0)

    # Split into splits
    split_scores = []
    for k in range(splits):
        part = preds[k * (N // splits): (k+1) * (N // splits)]
        py = part.mean(dim=0)  # marginal p(y)
        kl = part * (torch.log(part + 1e-10) - torch.log(py + 1e-10))
        kl_div = kl.sum(dim=1).mean()
        split_scores.append(torch.exp(kl_div).item())
    return np.mean(split_scores), np.std(split_scores)

In [ ]:
# =============================================================================
# FID FEATURES — 2048 numbers that "smell" like the photo
# =============================================================================
# Do NOT wrap Inception with nn.Sequential(*children()). Aux classifiers
# and skip paths break. Trick: replace the last Linear with Identity so
# forward() already returns the 2048-D pooled vector.
#

class InceptionFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        net = inception_v3(
            weights=Inception_V3_Weights.DEFAULT,
            transform_input=False,
            aux_logits=False,
        )
        net.fc = nn.Identity()  # was Linear(2048, 1000)
        net.eval()
        for p in net.parameters():
            p.requires_grad = False
        self.net = net

    def forward(self, x):
        return self.net(x)  # [N, 2048]


feature_extractor = InceptionFeatureExtractor().to(device)

# To extract features from a set of images:

def get_inception_features(images, batch_size=32):
    """
    images: tensor [N, 3, 299, 299] or list of PIL
    Returns numpy array of shape [N, 2048]
    """
    if isinstance(images, list):
        tensors = [inception_transform(img).unsqueeze(0) for img in images]
        images = torch.cat(tensors, dim=0)
    images = images.to(device)

    features = []
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = images[i:i+batch_size]
            feat = feature_extractor(batch)
            features.append(feat.cpu())
    return torch.cat(features, dim=0).numpy()

# Compute FID:
def calculate_fid(real_features, gen_features, eps=1e-6):
    """
    real_features, gen_features: numpy arrays of shape [N, D]
    Returns FID score.
    """
    mu_real = np.mean(real_features, axis=0)
    sigma_real = np.cov(real_features, rowvar=False)
    mu_gen = np.mean(gen_features, axis=0)
    sigma_gen = np.cov(gen_features, rowvar=False)

    diff = mu_real - mu_gen
    # Compute sqrt of product of covariances
    covmean, _ = linalg.sqrtm(sigma_real.dot(sigma_gen), disp=False)
    if not np.isfinite(covmean).all():
        offset = np.eye(sigma_real.shape[0]) * eps
        covmean = linalg.sqrtm((sigma_real + offset).dot(sigma_gen + offset))

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff.dot(diff) + np.trace(sigma_real + sigma_gen - 2*covmean)
    return float(fid)


In [ ]:
# CLIP Score Implementation
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Define a function to compute CLIP Score for a list of (image, text) pairs:

def clip_score(images, texts, batch_size=32):
    """
    images: list of PIL Images
    texts: list of strings (same length)
    Returns average CLIP Score (cosine similarity).
    """
    scores = []
    for i in range(0, len(images), batch_size):
        batch_images = images[i:i+batch_size]
        batch_texts = texts[i:i+batch_size]
        inputs = clip_processor(text=batch_texts, images=batch_images, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            outputs = clip_model(**inputs)
            img_embeds = outputs.image_embeds
            text_embeds = outputs.text_embeds
            # Normalize
            img_embeds = img_embeds / img_embeds.norm(dim=-1, keepdim=True)
            text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
            cos_sim = (img_embeds * text_embeds).sum(dim=-1)
            scores.extend(cos_sim.cpu().tolist())
    return np.mean(scores), np.std(scores)

In [ ]:
# =============================================================================
# TOY DATA — paint tiny pictures here (no downloads)
# =============================================================================
# No COCO. No CIFAR. No Stable Diffusion. Just numpy + PIL.
#
# Three piles:
#   real_images      = simple colored shapes (our "dataset")
#   generated_images = same shapes + grain          ("sloppy forger")
#   noise_images     = pure static                  ("not even trying")
#
# Expect: FID(real, real) tiny · FID(real, grainy) medium · FID(real, noise) huge
# N=16 is tiny → scores are noisy. Trust the ORDER, not the digits.
#

N = 16
SIZE = 64
rng = np.random.default_rng(42)

# (name, RGB) — CLIP captions use the name
PALETTE = [
    ("red circle", (220, 40, 40)),
    ("blue square", (40, 80, 220)),
    ("green circle", (40, 180, 70)),
    ("yellow square", (230, 200, 40)),
]


def paint_shape(name, rgb, rng):
    """Draw one circle or square on a light background."""
    canvas = np.full((SIZE, SIZE, 3), 240, dtype=np.uint8)
    yy, xx = np.ogrid[:SIZE, :SIZE]
    cy, cx = rng.integers(20, SIZE - 20, size=2)
    rad = int(rng.integers(12, 22))
    if "circle" in name:
        mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= rad**2
    else:
        mask = (np.abs(xx - cx) <= rad) & (np.abs(yy - cy) <= rad)
    canvas[mask] = rgb
    return Image.fromarray(canvas)


def add_grain(img, strength=45, rng=None):
    rng = np.random.default_rng(0) if rng is None else rng
    arr = np.asarray(img).astype(np.float32)
    arr = np.clip(arr + rng.normal(0.0, strength, arr.shape), 0, 255).astype(np.uint8)
    return Image.fromarray(arr)


real_images, prompts = [], []
for i in range(N):
    name, rgb = PALETTE[i % len(PALETTE)]
    real_images.append(paint_shape(name, rgb, rng))
    prompts.append(f"a photo of a {name}")

generated_images = [add_grain(im, rng=rng) for im in real_images]
noise_images = [
    Image.fromarray(rng.integers(0, 256, (SIZE, SIZE, 3), dtype=np.uint8))
    for _ in range(N)
]

print(f"Painted {len(real_images)} toy images (no download).")
print(f"Built {len(generated_images)} grainy fakes and {len(noise_images)} noise fakes.")
print("Captions (for CLIP):", prompts[:5], "...")

fig, axes = plt.subplots(3, 8, figsize=(12, 4.5))
rows = [("real", real_images), ("grainy fake", generated_images), ("noise", noise_images)]
for r, (lab, pile) in enumerate(rows):
    for c in range(8):
        axes[r, c].imshow(pile[c])
        axes[r, c].axis("off")
        if c == 0:
            axes[r, c].set_ylabel(lab, fontsize=8)
plt.suptitle("Toy piles for IS / FID / CLIP — drawn in-notebook, nothing downloaded")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# FID on the three piles
# =============================================================================
# get_inception_features already resizes PIL → 299×299 and runs the net.
# splits=2 / N=16: the NUMBER is noisy. Trust the ORDER, not the digits.
#

real_features = get_inception_features(real_images)
grain_features = get_inception_features(generated_images)
noise_features = get_inception_features(noise_images)

fid_self = calculate_fid(real_features, real_features)
fid_grain = calculate_fid(real_features, grain_features)
fid_noise = calculate_fid(real_features, noise_features)

print(f"FID real vs real   (same pile):     {fid_self:.2f}   ← want ~0")
print(f"FID real vs grainy (weak forger):    {fid_grain:.2f}   ← bigger")
print(f"FID real vs noise  (random static):  {fid_noise:.2f}   ← biggest")
print("HOW TO READ: order should be  self << grainy << noise. Tiny N → jumpy values.")

In [ ]:
# =============================================================================
# INCEPTION SCORE on the same piles
# =============================================================================
# splits must be ≤ N. With 16 images we only use 2 splits (still noisy).
#

is_real = inception_score(real_images, batch_size=8, splits=2)
is_grain = inception_score(generated_images, batch_size=8, splits=2)
is_noise = inception_score(noise_images, batch_size=8, splits=2)

print(f"IS real:   mean={is_real[0]:.2f}  std={is_real[1]:.2f}   ← CIFAR objects, often confident")
print(f"IS grainy: mean={is_grain[0]:.2f}  std={is_grain[1]:.2f}   ← blurrier → usually lower")
print(f"IS noise:  mean={is_noise[0]:.2f}  std={is_noise[1]:.2f}   ← mush → near 1")
print("HOW TO READ: HIGHER is better. Noise should sit near 1. Don't treat 16 images as a paper score.")

In [ ]:
# =============================================================================
# CLIP SCORE — does the picture match the WORDS?
# =============================================================================
# Same 16 photos, two caption lists:
#   prompts      = true CIFAR class  ("a photo of a cat")
#   wrong_prompts = one label for ALL ("a photo of a submarine")
#
# Real+true captions should beat real+wrong captions.
# Grainy still often matches the class. Noise should be closest to chance.
#

wrong_prompts = ["a photo of a submarine"] * len(real_images)

clip_real, clip_real_std = clip_score(real_images, prompts)
clip_wrong, clip_wrong_std = clip_score(real_images, wrong_prompts)
clip_grain, clip_grain_std = clip_score(generated_images, prompts)
clip_noise, clip_noise_std = clip_score(noise_images, prompts)

print(f"CLIP real + true labels:  {clip_real:.3f} ± {clip_real_std:.3f}   ← highest")
print(f"CLIP real + WRONG labels: {clip_wrong:.3f} ± {clip_wrong_std:.3f}   ← should drop")
print(f"CLIP grainy + true:      {clip_grain:.3f} ± {clip_grain_std:.3f}")
print(f"CLIP noise + true:       {clip_noise:.3f} ± {clip_noise_std:.3f}   ← lowest-ish")
print("HOW TO READ: HIGHER = image and text agree. This is the metric IS/FID cannot compute.")